# Hunga vapor-rich source comparison

This notebook evaluates vapor-rich Hunga source end members using the validated ThermoPlume vapor-only formulation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from thermoplume import ThermoPlumeModel, ThermoPlumeParameters


## Source cases

Steam mass fractions are mapped to ThermoPlume source loading using

$$\Omega = \frac{m_s}{1-m_s}$$

with volcanic-gas fraction within the non-steam source defined as

$$n = \frac{m_g}{1-m_s}.$$


In [ ]:
steam_fractions = np.array([0.0, 0.2, 0.5, 0.9])
m_g = 0.03

Omega = steam_fractions / (1.0 - steam_fractions)
n = m_g / (1.0 - steam_fractions)

source_cases = pd.DataFrame({
    'steam_fraction': steam_fractions,
    'Omega': Omega,
    'n': n,
})

source_cases


## Vapor-only neutral-buoyancy thresholds

In [ ]:
T_m = 1323.15
T_w = 373.15

rows = []

for row in source_cases.itertuples(index=False):
    p = ThermoPlumeParameters(
        Omega=row.Omega,
        n=row.n,
        T_m=T_m,
        T_w=T_w,
    )

    model = ThermoPlumeModel(p)

    analytical = model.solve(
        method='analytical',
        water_state='vapor',
    )

    python = model.solve(
        method='numerical',
        backend='python',
        water_state='vapor',
    )

    cpp = model.solve(
        method='numerical',
        backend='cpp',
        water_state='vapor',
    )

    rows.append({
        'steam_fraction': row.steam_fraction,
        'Omega': row.Omega,
        'n': row.n,
        'xi_analytical': analytical['xi_crit'],
        'xi_python': python['xi_crit'],
        'xi_cpp': cpp['xi_crit'],
    })

results = pd.DataFrame(rows)
results


In [ ]:
for row in results.itertuples(index=False):
    values = np.array([
        row.xi_analytical,
        row.xi_python,
        row.xi_cpp,
    ])

    finite = np.isfinite(values)

    if finite.any():
        assert finite.all()
        assert np.allclose(
            values,
            values[0],
            rtol=1e-8,
            atol=1e-10,
        )
    else:
        assert (~finite).all()

print('Hunga vapor cases agree across all three solvers')
